# Web Data Science
## Week 05 - Static web pages

INFO 4871/5871 - Fall 2024  
[Brian C. Keegan, Ph.D.](http://brianckeegan.com/)  
[Assistant Professor, Department of Information Science](https://www.colorado.edu/cmci/people/information-science/brian-c-keegan)  
University of Colorado Boulder  

Copyright and distributed under an [MIT License](https://opensource.org/licenses/MIT)

## Parsing HTML data into tabular data

The overall goal we have as researchers in scraping data from the web is converting data from one structured format (HTML's tree-like structures) into another structured format (probably a tabular structure with rows and columns). 
This could involve simply reading tables out of a webpage all the way up to taking irregularly-structured HTML elements into a tabular format. 

We are going to make some use of the [`pandas`](https://pandas.pydata.org/) library ("**pan**el **da**ta", not the cute animal), which is Python's implementation of a data frame concept. This is a very powerful and complex library that I typically spend more than 12 hours of lecture teaching in intermediate programming classes. I hope to convey some important elements as we work through material, but it is far beyond the scope of this class to be able to cover all the fundamentals and syntax. 

Let's begin by importing the libraries we'll need in this notebook: requests, BeautifulSoup, and pandas

In [3]:
# Most straight-forward way to import a library in Python
import requests

# BeautifulSoup is a module inside the "bs4" library, we only import the BeautifulSoup module
from bs4 import BeautifulSoup

# We import pandas but give the library a shortcut alias "pd" since we will call its functions so much
import pandas as pd

# Define a user-agent string you'll use throughout the notebook for all your requests
user_agent = {'user-agent':'info-5871/0.0 Web Data Science, brian.keegan@colorado.edu'}

### Reading an HTML table into Python

[The Numbers](http://www.the-numbers.com) is a popular source of data about movies' box office revenue numbers. Their daily domestic charts are HTML tables with the top-grossing movies for each day of the year, going back for several years. This [table](https://www.the-numbers.com/box-office-chart/daily/2018/12/25) for Christmas day in 2018 has coluns for the current week's ranking, previous week's ranking, name of movie, distributor, gross, change over the previous week, number of theaters, revenue per theater, total gross, and number of days since release. This looks like a fairly straightforward table that could be read directly into data frame-like structure.

Using the Inspect tool, we can see the table exists as a `<table border="0" ... align="CENTER">` element with child tags like `<tbody>` and `<tr>` (table row). Each `<tr>` has `<td>` which defines each of the cells and their content. For more on how HTML defines tables, check out [this tutoral](https://www.w3schools.com/html/html_tables.asp).

Using `requests` and `BeautifulSoup` we would get this webpage's HTML, turn it into soup, and then find the table (`<table>`) or the table rows (`<tr>`) and pull out their content.

In [4]:
# Make the request
xmas_bo_raw = requests.get(
    'https://www.the-numbers.com/box-office-chart/daily/2024/9/24',
    headers = user_agent).text

# Turn into soup, specify the HTML parser
xmas_bo_soup = BeautifulSoup(xmas_bo_raw,'html.parser')

# Use .find_all to retrieve all the tables in the page
xmas_bo_tables = xmas_bo_soup.find_all('table')

xmas_bo_tables

[<table class="pintable">
 <tr>
 <td class="previous"><a href="/box-office-chart/daily/2024/09/03">← Previous Chart</a></td>
 <td class="index"><a href="/box-office">Chart Index</a></td>
 <td class="next"><a href="/box-office-chart/daily/2024/09/05">Next Chart →</a></td>
 </tr>
 </table>,
 <table class="table-responsive display compact dataTable" id="box_office_daily_table" style="border:none; width:100%">
 <thead><tr><th> </th><th> </th><th>Movie Title</th><th>Distributor</th><th>Gross</th><th>%YD</th><th>%LW</th><th>Theaters</th><th>Per<br/>Theater</th><th>Total<br/>Gross</th><th>Days In<br/>Release</th></tr></thead><tbody>
 <tr>
 <td class="data" data-sort="1">1</td>
 <td class="data" data-sort="999">(-)</td>
 <td><b><a href="/movie/Beetlejuice-Beetlejuice-(2024)#tab=box-office">Beetlejuice Beetlejuice</a></b></td>
 <td><a href="/market/distributor/Warner-Bros">Warner Bros.</a></td>
 <td class="data">$2,466,902</td>
 <td class="data" data-sort="0"> </td>
 <td class="data" data-sort=

In [5]:
len(xmas_bo_tables)

2

It turns out there are two tables on the page, the first is a baby table consisting of the "Previous Chart", "Chart Index", and "Next Chart" at the top. We want the second table with all the data: `xmas_bo_tables[1]` returns the second chart (remember that Python is 0-indexed, so the first chart is at `xmas_bo_tables[0]`). With this table identified, we can do a second `find_all` to get the table rows inside it and we save it as `xmas_bo_trs`.

In [6]:
xmas_bo_trs = xmas_bo_tables[1].find_all('tr')

xmas_bo_trs

[<tr><th> </th><th> </th><th>Movie Title</th><th>Distributor</th><th>Gross</th><th>%YD</th><th>%LW</th><th>Theaters</th><th>Per<br/>Theater</th><th>Total<br/>Gross</th><th>Days In<br/>Release</th></tr>,
 <tr>
 <td class="data" data-sort="1">1</td>
 <td class="data" data-sort="999">(-)</td>
 <td><b><a href="/movie/Beetlejuice-Beetlejuice-(2024)#tab=box-office">Beetlejuice Beetlejuice</a></b></td>
 <td><a href="/market/distributor/Warner-Bros">Warner Bros.</a></td>
 <td class="data">$2,466,902</td>
 <td class="data" data-sort="0"> </td>
 <td class="data" data-sort="0"> </td>
 <td class="data" data-sort="4172">4,172</td>
 <td class="data chart_grey" data-sort="591">$591</td>
 <td class="data" data-sort="230938348">$230,938,348</td>
 <td class="data">19</td>
 </tr>,
 <tr>
 <td class="data" data-sort="2">2</td>
 <td class="data" data-sort="999">(-)</td>
 <td><b><a href="/movie/Transformers-One-(2024)#tab=box-office">Transformers One</a></b></td>
 <td><a href="/market/distributor/Paramount-P

In [7]:
len(xmas_bo_trs)

40

Let's inspect a few of these rows. The first row in our list of rows under `xmas_bo_trs` should be the header with the names of the columns.

In [8]:
xmas_bo_trs[0]

<tr><th> </th><th> </th><th>Movie Title</th><th>Distributor</th><th>Gross</th><th>%YD</th><th>%LW</th><th>Theaters</th><th>Per<br/>Theater</th><th>Total<br/>Gross</th><th>Days In<br/>Release</th></tr>

The next table row should be for Aquaman.

In [9]:
xmas_bo_trs[1]

<tr>
<td class="data" data-sort="1">1</td>
<td class="data" data-sort="999">(-)</td>
<td><b><a href="/movie/Beetlejuice-Beetlejuice-(2024)#tab=box-office">Beetlejuice Beetlejuice</a></b></td>
<td><a href="/market/distributor/Warner-Bros">Warner Bros.</a></td>
<td class="data">$2,466,902</td>
<td class="data" data-sort="0"> </td>
<td class="data" data-sort="0"> </td>
<td class="data" data-sort="4172">4,172</td>
<td class="data chart_grey" data-sort="591">$591</td>
<td class="data" data-sort="230938348">$230,938,348</td>
<td class="data">19</td>
</tr>

If we wanted to access the contents of this table row, we could use the `.contents` method to get a list of each of the `<td>` table cells, which (frustratingly) intersperses newline characters.

In [10]:
xmas_bo_trs[1].contents

['\n',
 <td class="data" data-sort="1">1</td>,
 '\n',
 <td class="data" data-sort="999">(-)</td>,
 '\n',
 <td><b><a href="/movie/Beetlejuice-Beetlejuice-(2024)#tab=box-office">Beetlejuice Beetlejuice</a></b></td>,
 '\n',
 <td><a href="/market/distributor/Warner-Bros">Warner Bros.</a></td>,
 '\n',
 <td class="data">$2,466,902</td>,
 '\n',
 <td class="data" data-sort="0"> </td>,
 '\n',
 <td class="data" data-sort="0"> </td>,
 '\n',
 <td class="data" data-sort="4172">4,172</td>,
 '\n',
 <td class="data chart_grey" data-sort="591">$591</td>,
 '\n',
 <td class="data" data-sort="230938348">$230,938,348</td>,
 '\n',
 <td class="data">19</td>,
 '\n']

Another alternative is to use the `.text` method to get the text content of all the cells in this row.

In [11]:
xmas_bo_trs[1].text

'\n1\n(-)\nBeetlejuice Beetlejuice\nWarner Bros.\n$2,466,902\n\xa0\n\xa0\n4,172\n$591\n$230,938,348\n19\n'

The `\n` characters re-appear here, but if we `print` out this statement, we see their newline functionality.

In [12]:
print(xmas_bo_trs[1].text)


1
(-)
Beetlejuice Beetlejuice
Warner Bros.
$2,466,902
 
 
4,172
$591
$230,938,348
19



We could use string processing to take this text string and convert it into a simple list of data. `.split('\n')` will split the string on the newline characters and return a list of what exists in between.

In [13]:
xmas_bo_trs[1].text.split('\n')

['',
 '1',
 '(-)',
 'Beetlejuice Beetlejuice',
 'Warner Bros.',
 '$2,466,902',
 '\xa0',
 '\xa0',
 '4,172',
 '$591',
 '$230,938,348',
 '19',
 '']

We'll write a `for` loop to go through all the table rows in `xmas_bo_trs`, get the list of data from the row, and add it back to a list of all the rows.

In [14]:
cleaned_xmas_bo_rows = []

# Loop through all the non-header (first row) table rows
for row in xmas_bo_trs[1:]:
    
    # Get the text of the row and split on the newlines (like above)
    cleaned_row = row.text.split('\n')
    
    # Add this cleaned row back to the external list of row data
    cleaned_xmas_bo_rows.append(cleaned_row)
    
# Inspect the first few rows of data
cleaned_xmas_bo_rows[:2]

[['',
  '1',
  '(-)',
  'Beetlejuice Beetlejuice',
  'Warner Bros.',
  '$2,466,902',
  '\xa0',
  '\xa0',
  '4,172',
  '$591',
  '$230,938,348',
  '19',
  ''],
 ['',
  '2',
  '(-)',
  'Transformers One',
  'Paramount Pi…',
  '$2,003,977',
  '\xa0',
  '\xa0',
  '3,978',
  '$504',
  '$27,871,915',
  '5',
  '']]

In [15]:
len(cleaned_xmas_bo_rows)

39

Now we can pass this list of lists in `cleaned_xmas_bo_rows` to pandas's `DataFrame` function and hopefully get a nice table out.

In [16]:
xmas_bo_df = pd.DataFrame(cleaned_xmas_bo_rows)

# Inspect
xmas_bo_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,,1,(-),Beetlejuice Beetlejuice,Warner Bros.,"$2,466,902",,,"4,172",$591,"$230,938,348",19,
1,,2,(-),Transformers One,Paramount Pi…,"$2,003,977",,,"3,978",$504,"$27,871,915",5,
2,,3,(-),Speak No Evil,Universal,"$856,980",,,"3,375",$254,"$22,752,895",12,
3,,4,(-),The Substance,MUBI,"$611,797",,,"1,949",$314,"$4,239,480",5,
4,,5,(-),Never Let Go,Lionsgate,"$585,690",,,"2,667",$220,"$5,409,643",5,


We need to do a bit of cleanup on this data:

* Columns 0 and 12 are all empty
* Add column names

In [17]:
# Drop columns 0 and 12 and overwrite the xmas_box_df variable
xmas_bo_df = xmas_bo_df.drop(columns=[0,12])

# Rename the columns
xmas_bo_df.columns = ['Rank','Last rank','Movie','Distributor','Gross',
                      'Change','LW','Theaters','Per theater','Total gross',
                      'Days']

# Write to disk
# xmas_bo_df.to_csv('christmas_2018_box_office.csv',encoding='utf8')

# Inspect
xmas_bo_df.head()

,Rank,Last rank,Movie,Distributor,Gross,Change,LW,Theaters,Per theater,Total gross,Days
0,1,(-),Beetlejuice Beetlejuice,Warner Bros.,"$2,466,902",,,"4,172",$591,"$230,938,348",19
1,2,(-),Transformers One,Paramount Pi…,"$2,003,977",,,"3,978",$504,"$27,871,915",5
2,3,(-),Speak No Evil,Universal,"$856,980",,,"3,375",$254,"$22,752,895",12
3,4,(-),The Substance,MUBI,"$611,797",,,"1,949",$314,"$4,239,480",5
4,5,(-),Never Let Go,Lionsgate,"$585,690",,,"2,667",$220,"$5,409,643",5


### `pandas`'s `read_html`
That was a good amount of work just to get this simple HTML table into Python. But it was important to cover how table elements moved from a string in `requests`, into a soup object from `BeautifulSoup`. into a list of data, and finally into `pandas`. 

`pandas` also has powerful functionality for reading tables directly from HTML. If we convert the soup of the first table (`xmas_bo_tables[1]`) back into a string, `pandas` can read it directly into a table. 

There are a few ideosyncracies here, the result is a list of dataframes—even if there's only a single table/dataframe—so we need to return the first (and only) element of this list. This is why there's a `[0]` at the end and the `.head()` is just to show the first five rows.

In [22]:
str(xmas_bo_tables[1])

'<table class="table-responsive display compact dataTable" id="box_office_daily_table" style="border:none; width:100%">\n<thead><tr><th>\xa0</th><th>\xa0</th><th>Movie Title</th><th>Distributor</th><th>Gross</th><th>%YD</th><th>%LW</th><th>Theaters</th><th>Per<br/>Theater</th><th>Total<br/>Gross</th><th>Days In<br/>Release</th></tr></thead><tbody>\n<tr>\n<td class="data" data-sort="1">1</td>\n<td class="data" data-sort="999">(-)</td>\n<td><b><a href="/movie/Beetlejuice-Beetlejuice-(2024)#tab=box-office">Beetlejuice Beetlejuice</a></b></td>\n<td><a href="/market/distributor/Warner-Bros">Warner Bros.</a></td>\n<td class="data">$2,466,902</td>\n<td class="data" data-sort="0">\xa0</td>\n<td class="data" data-sort="0">\xa0</td>\n<td class="data" data-sort="4172">4,172</td>\n<td class="data chart_grey" data-sort="591">$591</td>\n<td class="data" data-sort="230938348">$230,938,348</td>\n<td class="data">19</td>\n</tr>\n<tr>\n<td class="data" data-sort="2">2</td>\n<td class="data" data-sort="9

In [24]:
xmas_bo_table_as_string = str(xmas_bo_tables[1])

pd.read_html(xmas_bo_table_as_string)[0]

/var/folders/lr/v195xr617d32k2zdwwh30sb40000gn/T/ipykernel_57110/1286638848.py:3: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_html(xmas_bo_table_as_string)[0]


,Unnamed: 0,Unnamed: 1,Movie Title,Distributor,Gross,%YD,%LW,Theaters,Per Theater,Total Gross,Days In Release
0,1,(-),Beetlejuice Beetlejuice,Warner Bros.,"$2,466,902",NaN,NaN,4172.0,$591,"$230,938,348",19.0
1,2,(-),Transformers One,Paramount Pi…,"$2,003,977",NaN,NaN,3978.0,$504,"$27,871,915",5.0
2,3,(-),Speak No Evil,Universal,"$856,980",NaN,NaN,3375.0,$254,"$22,752,895",12.0
3,4,(-),The Substance,MUBI,"$611,797",NaN,NaN,1949.0,$314,"$4,239,480",5.0
4,5,(-),Never Let Go,Lionsgate,"$585,690",NaN,NaN,2667.0,$220,"$5,409,643",5.0
5,6,(6),Deadpool & Wolverine,Walt Disney,"$415,308",-68%,-74%,2450.0,$170,"$627,950,324",61.0
6,7,(8),Reagan,ShowBiz Direct,"$345,565",-73%,NaN,1850.0,$187,"$27,076,790",26.0
7,8,(-),Am I Racist?,SDG Releasing,"$333,907",NaN,NaN,1600.0,$209,"$9,534,617",12.0
8,9,(10),The Forge,Sony Pictures,"$189,784",-73%,-71%,1181.0,$161,"$26,637,232",33.0
9,10,(-),The Killer’s Game,Lionsgate,"$188,157",NaN,NaN,2623.0,$72,"$5,124,785",12.0


The column names got lumped in as rows, but we can fix this as well with the `read_html` function by passing the row index where the column lives. In this case, it is the first row, so we pass `header=0`.

In [ ]:
pd.read_html(xmas_bo_table_as_string,header=0)[0].head()

Finally, you can point `read_html` at a URL without any `requests` or `BeautifulSoup` and get all the tables on the page as a list of DataFrames. `pandas` is simply doing the `requests` and `BeautifulSoup` on the inside. Interestingly, I'm getting a [HTTP 403](https://en.wikipedia.org/wiki/HTTP_403) error indicating the server (The Numbers) is forbidding the client (us) from accessing their data using this strategy. We will discuss next week whether and how to handle situations where web servers refuse connections from non-human clients. In this case, you cannot use the off-the-shelf `read_html` approach and would need to revert to using the `requests`+`BeautifulSoup` approach above.

In [25]:
simple_tables = pd.read_html(
    'https://www.the-numbers.com/box-office-chart/daily/2018/12/25',
)
simple_tables

HTTPError: HTTP Error 403: Forbidden

If we point it at Wikipedia's [2018 in film](https://en.wikipedia.org/wiki/2018_in_film), it will pull all of the tables present on the page.

In [26]:
simple_tables = pd.read_html('https://en.wikipedia.org/wiki/2018_in_film')
len(simple_tables)

9

The first three correspond to the "Year in film" navigation box on the side and are poorly-formatted by default.

In [30]:
simple_tables[3]

,Rank,Title,Distributor,Worldwide gross
0,1,Avengers: Infinity War,Disney,"$2,048,359,754"
1,2,Black Panther,Disney,"$1,347,597,973"
2,3,Jurassic World: Fallen Kingdom,Universal,"$1,308,467,944"
3,4,Incredibles 2,Disney,"$1,242,805,359"
4,5,Aquaman,Warner Bros.,"$1,148,528,393"
5,6,Bohemian Rhapsody,20th Century Fox,"$910,809,311"
6,7,Venom,Sony,"$856,085,151"
7,8,Mission: Impossible – Fallout,Paramount,"$791,115,104"
8,9,Deadpool 2,20th Century Fox,"$785,896,609"
9,10,Fantastic Beasts: The Crimes of Grindelwald,Warner Bros.,"$654,855,901"


The fourth table in the `simple_tables` list we got from parsing the Wikipedia page with `read_html` is the table under the "Highest-grossing films" section.

In [ ]:
simple_tables[3]

You can pass the "header" option in `read_html` to make sure the column names from a particular row (in this case the first row) do not accidentally become rows of data. 

In [31]:
wiki_top_grossing_t = pd.read_html('https://en.wikipedia.org/wiki/2018_in_film',header=0)[3]
wiki_top_grossing_t

,Rank,Title,Distributor,Worldwide gross
0,1,Avengers: Infinity War,Disney,"$2,048,359,754"
1,2,Black Panther,Disney,"$1,347,597,973"
2,3,Jurassic World: Fallen Kingdom,Universal,"$1,308,467,944"
3,4,Incredibles 2,Disney,"$1,242,805,359"
4,5,Aquaman,Warner Bros.,"$1,148,528,393"
5,6,Bohemian Rhapsody,20th Century Fox,"$910,809,311"
6,7,Venom,Sony,"$856,085,151"
7,8,Mission: Impossible – Fallout,Paramount,"$791,115,104"
8,9,Deadpool 2,20th Century Fox,"$785,896,609"
9,10,Fantastic Beasts: The Crimes of Grindelwald,Warner Bros.,"$654,855,901"


In [32]:
wiki_top_grossing_t['Worldwide gross']

0    $2,048,359,754
1    $1,347,597,973
2    $1,308,467,944
3    $1,242,805,359
4    $1,148,528,393
5      $910,809,311
6      $856,085,151
7      $791,115,104
8      $785,896,609
9      $654,855,901
Name: Worldwide gross, dtype: object

Note that there are still a few errors in this table because the "Disney" value in the Wikipedia table spans two rows and `read_html` thus skips the "Distributor" value for Black Panther. 

In [ ]:
# Copy the value at index position 1, column position Distributor to Wordwide gross
wiki_top_grossing_t.loc[1,'Worldwide gross'] = wiki_top_grossing_t.loc[1,'Distributor']

# Change the value at 1, Distributor to Disney
wiki_top_grossing_t.loc[1,'Distributor'] = 'Disney'

wiki_top_grossing_t

### Exercise 01

Grab a table from Wikipedia using pandas `read_html`.

In [38]:
wiki_mountains = pd.read_html('https://en.wikipedia.org/wiki/List_of_highest_mountains_on_Earth')

wiki_mountains[1]

,Rank[i],Mountain name(s),Height (rounded)[ii],Prominence (rounded)[iii],Range,Coordinates[iv],Parent mountain [v],First ascent[vi],Country,Photo
0,1,Mount EverestSagarmathaChomolungma,"8,849 metres (29,032 ft) [a]","8,849 metres (29,032 ft)",Mahalangur Himalaya,27°59′17″N 86°55′30″E﻿ / ﻿27.9881°N 86.925°E,—,1953,NepalChina,NaN
1,2,K2,"8,611 metres (28,251 ft)","4,020 metres (13,190 ft)",Baltoro Karakoram,35°52′53″N 76°30′48″E﻿ / ﻿35.88139°N 76.51333°E,Mount Everest,1954,Pakistan[b]China[16],NaN
2,3,Kangchenjunga,"8,586 metres (28,169 ft)","3,922 metres (12,867 ft)",Kangchenjunga Himalaya,27°42′12″N 88°08′51″E﻿ / ﻿27.70333°N 88.14750°E *,Mount Everest,1955,NepalIndia,NaN
3,4,Lhotse,"8,516 metres (27,940 ft)","610 metres (2,000 ft)",Mahalangur Himalaya,27°57′42″N 86°55′59″E﻿ / ﻿27.96167°N 86.93306°E,Mount Everest,1956,ChinaNepal,NaN
4,5,Makalu,"8,485 metres (27,838 ft)","2,378 metres (7,802 ft)",Mahalangur Himalaya,27°53′23″N 87°05′20″E﻿ / ﻿27.88972°N 87.08889°E,Mount Everest,1955,NepalChina,NaN
...,...,...,...,...,...,...,...,...,...,...
115,104,Noijin KangsangNorin Kang,"7,206 metres (23,642 ft)","2,160 metres (7,090 ft)",Nagarze Himalaya,28°56′48″N 90°10′42″E﻿ / ﻿28.94667°N 90.17833°E,Gangkhar Puensum,1986,China,NaN
116,105,Langtang Ri,"7,205 metres (23,638 ft)","665 metres (2,182 ft)",Langtang Himalaya,28°22′53″N 85°41′01″E﻿ / ﻿28.38139°N 85.68361°E,Shishapangma,1981,NepalChina,NaN
117,106,Kangphu KangShimokangri,"7,204 metres (23,635 ft)","1,244 metres (4,081 ft)",Lunana Himalaya,28°09′24″N 90°04′15″E﻿ / ﻿28.15667°N 90.07083°E,Tongshanjiabu,2002,BhutanChina[k],NaN
118,107,Singhi Kangri,"7,202 metres (23,629 ft)","730 metres (2,400 ft)",Siachen Karakoram,35°35′59″N 76°59′01″E﻿ / ﻿35.59972°N 76.98361°E,Teram Kangri III,1976,IndiaChina[g][h][f],NaN


In [34]:
got_s8_tables[1]

,No. overall,No. in season,Title,Directed by,Written by,Original air date [2],U.S. viewers (millions),Unnamed: 7
0,68,1,"""Winterfell""",David Nutter,Dave Hill,"April 14, 2019",11.76[3],NaN
1,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...,Upon reaching Winterfell with their combined a...
2,69,2,"""A Knight of the Seven Kingdoms""",David Nutter,Bryan Cogman,"April 21, 2019",10.29[4],NaN
3,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...,Jaime reveals Cersei's deception to the Targar...
4,70,3,"""The Long Night""",Miguel Sapochnik,David Benioff & D. B. Weiss,"April 28, 2019",12.02[5],NaN
5,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...,The living meet the Army of the Dead outside W...
6,71,4,"""The Last of the Starks""",David Nutter,David Benioff & D. B. Weiss,"May 5, 2019",11.80[6],NaN
7,"The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c...","The survivors mourn, burn the dead, and then c..."
8,72,5,"""The Bells""",Miguel Sapochnik,David Benioff & D. B. Weiss,"May 12, 2019",12.48[7],NaN
9,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...,Varys urges Jon to advance his claim to the th...


## Writing your own parser

We will return to the historical Oscars data. Even though data as prominent as this is likely to already exist in tabular format somewhere, we will maintain the illusion that we are the first to both scrape it and parse it into a tabular format. Our goal here is to write a parser that will (ideally) work across multiple pages; in this case, each of the award years.

One of the first things we should do before writing any code is come up with a model of what we want our data to look like at the end of this. This is an intuitive and "tidy" format, but you might come up with alternatives based on your analysis and modeling needs.

| *Year* | *Category* | *Nominee* | *Movie* | *Won* |
| --- | --- | --- | --- | --- |
| 2019 | Actor in a leading role | Christian Bale | Vice | NA |
| 2019 | Actor in a leading role | Bradley Cooper | A Star Is Born | NA |
| 2019 | Actor in a leading role | Willem Dafoe | At Eternity's Gate | NA |
| 2019 | Actor in a leading role | Rami Malek | Bohemian Rhapsody | NA |
| 2019 | Actor in a leading role | Viggo Mortensen | Green Book | NA |

We will begin with writing a parser for a (hopefully!) representative year, then scrape the data for all the years, then apply the scraper to each of those years, and finally combine all the years' data together into a large data set. 

Let's begin with writing a parser for a (hopefully!) representative year. We will use 2018 instead and make the profoundly naïve assumption it should work the same going back in time.

Start off with using `requests` to get the data and then use `BeautifulSoup` to turn it into soup we can parse through.

In [39]:
user_agent

{'user-agent': 'info-5871/0.0 Web Data Science, brian.keegan@colorado.edu'}

Using the Inspect tool exercise from Class 1, the `<div class="view-grouping">` seems to be the most promising tag for us to extract. Use `.find_all('div',{'class':'field__item'})` to (hopefully!) get all of these award groups. Inspect the first and last ones to make sure they looks coherent.

In [40]:
oscars_raw = requests.get('https://www.oscars.org/oscars/ceremonies/2024',headers=user_agent).text

oscars_soup = BeautifulSoup(oscars_raw)

In [41]:
# Get all the groups that have a <div class="view-grouping"> tag
oscars_groups = oscars_soup.find_all('div',{'class':'field__item'})

# Inspect the first one
oscars_groups[0]

<div class="field field--name-field-location field--type-string field--label-hidden field__item">Dolby Theatre at Ovation Hollywood</div>

The last group is something besides "Writing (Original Screenplay)" and it's not clear to me where this tag's content renders on the page.

In [42]:
# Inspect the last one
oscars_groups[-1]

<div class="field__item">Written by Celine Song</div>

This puts us into something of a bind going forward: if the `.find_all` returns more groupings than we expected, then it's not sufficiently precise to identify *only* groupings of nominees.  However, there do not appear to be any child tags in the `oscars2018_groups[0]` grouping that uniquely differentiate them from the child tags present in the `oscars2018_groups[-1]` grouping. Another alternative is to simple parse the first 24 groupings, but this is a very brittle solution since other years' awards might have more or fewer groupings.

In [43]:
len(oscars_groups)

432

### Navigating the HTML tree to find more specific parent elements
A third alternative is to leverage the tree structure of HTML and get the parent element in the hopes it is more unique than its children. In this case something like `<div class="field field--name-field-award-categories field--type-entity-reference-revisions field--label-hidden field__items"...>` is a promising lead. Use `find_all` to search for this tag and confirm there is only one the one `<div>` element (with its children) rather than multiple `<div>` elements matching "quicktabs-container-honorees".

In [44]:
# Get the new tag group
oscars_parent_group = oscars_soup.find_all('div',{'class':'field field--name-field-award-categories field--type-entity-reference-revisions field--label-hidden field__items'})

# Hopefully there is only one group matching this pattern
len(oscars_parent_group)


1

So far so good, now we can use `find_all` on the soup for this `<div class="field__item">` to search *within* this specific parent group and hopefully there should be the 24 awards groupings. Nope, still lots.

In [45]:
# Note the addition of the [0] since the _parent_group is a list with 1 element in it
# We just extract that single element (which is a soup) and then we can use find_all on it

oscars_child_groups = oscars_parent_group[0].find_all('div',{'class':'field__item'})

len(oscars_child_groups)

406

Hallelujah! The award names for each group live inside a `<div class="field field--name-field-award-honorees field--type-entity-reference-revisions field--label-hidden field__items">`, so we can `find_all` for those, loop through each, and print out the name.

In [46]:
oscars_award_groups = oscars_parent_group[0].find_all('div',{'class':'paragraph paragraph--type--award-category paragraph--view-mode--default'})
len(oscars_award_groups)

23

In [47]:
for group in oscars_award_groups:
    print(group.text)


Actor in a Leading Role



Winner


Cillian Murphy



Oppenheimer





Nominees


Bradley Cooper



Maestro





Nominees


Colman Domingo



Rustin





Nominees


Paul Giamatti



The Holdovers





Nominees


Jeffrey Wright



American Fiction






Actor in a Supporting Role



Winner


Robert Downey Jr.



Oppenheimer





Nominees


Sterling K. Brown



American Fiction





Nominees


Robert De Niro



Killers of the Flower Moon





Nominees


Ryan Gosling



Barbie





Nominees


Mark Ruffalo



Poor Things






Actress in a Leading Role



Winner


Emma Stone



Poor Things





Nominees


Annette Bening



Nyad





Nominees


Lily Gladstone



Killers of the Flower Moon





Nominees


Sandra Hüller



Anatomy of a Fall





Nominees


Carey Mulligan



Maestro






Actress in a Supporting Role



Winner


Da'Vine Joy Randolph



The Holdovers





Nominees


Emily Blunt



Oppenheimer





Nominees


Danielle Brooks



The Color Purple





Nominees


America Ferrera



### Navigating the HTML tree from a specific child to find specific generic parents
Now bear with me through some additional and presently unnecessary pain. Above, we were able to isolate the 24 category groupings we wanted through finding an appropriate *parent* tag and then working *down*. But I also want to show how we could identify the same 24 category groups by finding an appropriate *child* tag and working back up. This could be helpful in other situations where the elements are hard to disambiguate.

Let's start by finding the tag for the "Actor in a Leading Role" from the soup containing all the tags.

In [ ]:
oscars_award_groups[0]

We could go up to get the parent and then find all 24 of the `<div class='view-grouping'>` among the children.

In [ ]:
# From the child we like, get its parent
oscars2018_group0_parent = oscars_parent_group[0].parent

# Now with the parent, find all the relevant children
oscars2018_group0_parent_children = oscars2018_group0_parent.find_all('div',{'class':'view-grouping'})

# Confirm
len(oscars2018_group0_parent_children)

### Checking the relevant fields

That seemed like a major digression away from the core task of writing a parser, but it is critical that we write a parser that parses *only* the data we want and nothing else. Now that we have our 24 awards groups in `oscars2018_true_groups`, let's break one open and extract all the yummy data waiting inside.

There are a few `<div>` sub-classes that are helpfully named that should make extracting this data a bit easier.

* `<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">` - name of the category
* `<div class="field field--name-field-honoree-type winner">` - winner
* `<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">` - name of actor
* `<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">` - title of movie

In [59]:
oscars_award_groups[0]

<div class="paragraph paragraph--type--award-category paragraph--view-mode--default" data-term-id="1731">
<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">Actor in a Leading Role</div>
<div class="field field--name-field-award-honorees field--type-entity-reference-revisions field--label-hidden field__items">
<div class="field__item">
<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
<div class="field field--name-field-honoree-type winner">Winner</div>
<div class="field">
<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
<div class="field__item">Cillian Murphy</div>
</div>
</div>
<div class="field">
<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
</div>
</div>
</div>
<div class="field__item">
<div class="paragraph paragraph--type--award-honor

"Zoom in" to the category name.

In [56]:
oscars_award_groups[-1].find_all('div',{'class':'field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item'})

[<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">Writing (Original Screenplay)</div>]

"Zoom in" to the actors' names.

In [57]:
oscars_award_groups[-1].find_all('div',{'class':'field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items'})

[<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Screenplay - Justine Triet and Arthur Harari</div>
 </div>,
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Written by David Hemingson</div>
 </div>,
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Written by Bradley Cooper &amp; Josh Singer</div>
 </div>,
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Screenplay by Samy Burch; Story by Samy Burch &amp; Alex Mechanik</div>
 </div>,
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Written by Celine Song</div>
 </div>]

In [58]:
oscars_award_groups[-1].find_all('div',{'class':'field field--name-field-honoree-type winner'})

[<div class="field field--name-field-honoree-type winner">Winner</div>]

### Writing the core parser functionality

How will we map the contents of the HTML to the 

* **Year**: We'll get this from the URL
* **Category**: `'field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item'`
* **Nominee**: `'field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items'`
* **Movie**: `'field field--name-field-award-film field--type-entity-reference field--label-hidden field__item'`
* **Won**: `'field field--name-field-honoree-type winner'`

In [60]:
oscars_award_groups[0]

<div class="paragraph paragraph--type--award-category paragraph--view-mode--default" data-term-id="1731">
<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">Actor in a Leading Role</div>
<div class="field field--name-field-award-honorees field--type-entity-reference-revisions field--label-hidden field__items">
<div class="field__item">
<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
<div class="field field--name-field-honoree-type winner">Winner</div>
<div class="field">
<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
<div class="field__item">Cillian Murphy</div>
</div>
</div>
<div class="field">
<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
</div>
</div>
</div>
<div class="field__item">
<div class="paragraph paragraph--type--award-honor

In [61]:
category = oscars_award_groups[0].find_all('div',{'class':'field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item'})[0].text
print("The name of the category is:",category)

The name of the category is: Actor in a Leading Role


In [68]:
names = []
nominees = oscars_award_groups[0].find_all('div',{'class':'field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items'})

for nominee in nominees:
    nominee_name = nominee.text
    names.append(nominee_name)
    print("The name of a nominee is:",nominee_name)

The name of a nominee is: 
Cillian Murphy

The name of a nominee is: 
Bradley Cooper

The name of a nominee is: 
Colman Domingo

The name of a nominee is: 
Paul Giamatti

The name of a nominee is: 
Jeffrey Wright



In [67]:
movie_names = []
movies = oscars_award_groups[0].find_all('div',{'class':'field field--name-field-award-film field--type-entity-reference field--label-hidden field__item'})

for movie in movies:
    if len(_movie.text) > 0:
        movie_name = movie.text.strip()
        movies.append(movie_name)
        print("The name of a movie is:",movie_name)

The name of a movie is: Oppenheimer
The name of a movie is: Maestro
The name of a movie is: Rustin
The name of a movie is: The Holdovers
The name of a movie is: American Fiction


# WE MADE IT THIS FAR ON TUESDAY

Another strategy is to use the `<div class="field__item">`s for each nominee and extract the relevant information from its subdivs. This is a bit more intuitive in the sense of reading from top to bottom and also makes it easier to capture the winner and losers based on position. But again the `field__item` tags are scattered everywhere and are too non-specific to be useful or reliable.

In [76]:
len(oscars_award_groups[0].find_all('div',{'class':'field__item'}))

16

In [79]:
oscars_award_groups[0].find_all('div',{'class':'field__item'})

[<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">Actor in a Leading Role</div>,
 <div class="field__item">
 <div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
 <div class="field field--name-field-honoree-type winner">Winner</div>
 <div class="field">
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Cillian Murphy</div>
 </div>
 </div>
 <div class="field">
 <div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
 </div>
 </div>
 </div>,
 <div class="field__item">Cillian Murphy</div>,
 <div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>,
 <div class="field__item">
 <div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
 <

The first `field__item` is the name of the category:

In [109]:
oscars_award_groups[0].find_all('div',{'class':'field__item'})[0]

<div class="field field--name-field-award-category-oscars field--type-entity-reference field--label-hidden field__item">Actor in a Leading Role</div>

The second `field__item` is the winner tag which has sub-tags that are also `field_item`s.

In [110]:
oscars_award_groups[0].find_all('div',{'class':'field__item'})[1]

<div class="field__item">
<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
<div class="field field--name-field-honoree-type winner">Winner</div>
<div class="field">
<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
<div class="field__item">Cillian Murphy</div>
</div>
</div>
<div class="field">
<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
</div>
</div>
</div>

The last `field__item` should be Jeffrey Wright, but it's a sub `field__item` about the movie that exists withing the larger string "field field--name-field-award-film field--type-entity-reference field--label-hidden **field__item**".

In [111]:
oscars_award_groups[0].find_all('div',{'class':'field__item'})[-1]

<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">American Fiction</div>

This is a good lesson that BeautifulSoup's `find_all` function picks up *any* of the text fields in the "class" value. Let's test whether we can just search for a value like `paragraph--type--award-honoree` instead of the whole gigantic string.

In [78]:
oscars_award_groups[0].find_all('div',{'class':'paragraph--type--award-honoree'})

[<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
 <div class="field field--name-field-honoree-type winner">Winner</div>
 <div class="field">
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Cillian Murphy</div>
 </div>
 </div>
 <div class="field">
 <div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
 </div>
 </div>,
 <div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
 <div class="field field--name-field-honoree-type nominee">Nominees</div>
 <div class="field">
 <div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
 <div class="field__item">Bradley Cooper</div>
 </div>
 </div>
 <div class="field">
 <div class="field field--name-field-award-film field--type-entity-reference field--label-hidden f

Does the first one have all the information about the first nominee? Winner, Movie, Name?

In [80]:
oscars_award_groups[0].find_all('div',{'class':'paragraph--type--award-honoree'})[0]

<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
<div class="field field--name-field-honoree-type winner">Winner</div>
<div class="field">
<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
<div class="field__item">Cillian Murphy</div>
</div>
</div>
<div class="field">
<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">Oppenheimer</div>
</div>
</div>

Does the last one have all the information: winner, movie, name?

In [112]:
oscars_award_groups[0].find_all('div',{'class':'paragraph--type--award-honoree'})[-1]

<div class="paragraph paragraph--type--award-honoree paragraph--view-mode--oscars">
<div class="field field--name-field-honoree-type nominee">Nominees</div>
<div class="field">
<div class="field field--name-field-award-entities field--type-entity-reference field--label-hidden field__items">
<div class="field__item">Jeffrey Wright</div>
</div>
</div>
<div class="field">
<div class="field field--name-field-award-film field--type-entity-reference field--label-hidden field__item">American Fiction</div>
</div>
</div>

In [115]:
nominees = oscars_award_groups[0].find_all('div',{'class':'paragraph--type--award-honoree'})

Use the `.find` method on each nominee's tag to pull out the relevant information:

* Winner: `field--name-field-honoree-type`
* Name: `field--name-field-award-entities`
* Film: `field--name-field-award-film`

Use the `.strip()` method to clear out any wacky characters or newlines hiding at the beginning or end of the string.

In [116]:
nominees[0].find('div',{'class':'field--name-field-honoree-type'}).text.strip()

'Winner'

In [117]:
nominees[0].find('div',{'class':'field--name-field-award-entities'}).text.strip()

'Cillian Murphy'

In [118]:
nominees[0].find('div',{'class':'field--name-field-award-film'}).text.strip()

'Oppenheimer'

Put it all together for all the nominees.

In [119]:
nominees = oscars_award_groups[0].find_all('div',{'class':'paragraph--type--award-honoree'})

# Loop through each nominee
for nominee in nominees:
    
    # Get the honoree type
    nom_winner = nominee.find('div',{'class':'field--name-field-honoree-type'}).text.strip()
    
    # Get the name
    nom_name = nominee.find('div',{'class':'field--name-field-award-entities'}).text.strip()
    
    # Get the movie
    nom_movie = nominee.find('div',{'class':'field--name-field-award-film'}).text.strip()
    
    print("{0} was nominated for \"{1}\" and was a {2}.".format(nom_name,nom_movie,nom_winner))

Cillian Murphy was nominated for "Oppenheimer" and was a Winner.
Bradley Cooper was nominated for "Maestro" and was a Nominees.
Colman Domingo was nominated for "Rustin" and was a Nominees.
Paul Giamatti was nominated for "The Holdovers" and was a Nominees.
Jeffrey Wright was nominated for "American Fiction" and was a Nominees.


Check that reversing "movie" and "name" works for another award category like original screenplay (`oscars_award_groups[-1]`). There's some weirdness with "Written by" and "Story by" filtering in here rather than simply names that may need to get fixed in the final calculation, but I would want to talk to a domain expert about the differences between these labels.

In [120]:
nominees = oscars_award_groups[-1].find_all('div',{'class':'paragraph--type--award-honoree'})

# Loop through each nominee
for nominee in nominees:
    
    # Get the honoree type
    nom_winner = nominee.find('div',{'class':'field--name-field-honoree-type'}).text.strip()
    
    # Get the name
    nom_name = nominee.find('div',{'class':'field--name-field-award-entities'}).text.strip()
    
    # Get the movie
    nom_movie = nominee.find('div',{'class':'field--name-field-award-film'}).text.strip()
    
    print("{0} was nominated for \"{1}\" and was a {2}.".format(nom_name,nom_movie,nom_winner))

Screenplay - Justine Triet and Arthur Harari was nominated for "Anatomy of a Fall" and was a Winner.
Written by David Hemingson was nominated for "The Holdovers" and was a Nominees.
Written by Bradley Cooper & Josh Singer was nominated for "Maestro" and was a Nominees.
Screenplay by Samy Burch; Story by Samy Burch & Alex Mechanik was nominated for "May December" and was a Nominees.
Written by Celine Song was nominated for "Past Lives" and was a Nominees.


This was just for Best Actors, now lets add another layer for all the different awards categories. We can see the movie name and awardee switch is important now since most of the categories are reversed.

The nominee category is saved in: `field--name-field-honoree-type`

In [104]:
oscars_award_groups[0].find('div',{'class':'field--name-field-award-category-oscars'}).text.strip()

'Actor in a Leading Role'

Let's build on our previous loop and add an outer layer to loop through all the nominees.

In [121]:
# For each award group
for group in oscars_award_groups:

    # Get the name of the category
    category = group.find('div',{'class':'field--name-field-award-category-oscars'}).text.strip()

    # Get all the nominees in the group
    nominees = group.find_all('div',{'class':'paragraph--type--award-honoree'})
    
    # Loop through each nominee
    for nominee in nominees:
        
        # Get the honoree type
        nom_winner = nominee.find('div',{'class':'field--name-field-honoree-type'}).text.strip()
        
        # Get the name
        nom_name = nominee.find('div',{'class':'field--name-field-award-entities'}).text.strip()
        
        # Get the movie
        nom_movie = nominee.find('div',{'class':'field--name-field-award-film'}).text.strip()

        # Print it out
        print("{0} was nominated in {1} for \"{2}\" and was a {3}.".format(nom_name,category,nom_movie,nom_winner))

    print("\n")

Cillian Murphy was nominated in Actor in a Leading Role for "Oppenheimer" and was a Winner.
Bradley Cooper was nominated in Actor in a Leading Role for "Maestro" and was a Nominees.
Colman Domingo was nominated in Actor in a Leading Role for "Rustin" and was a Nominees.
Paul Giamatti was nominated in Actor in a Leading Role for "The Holdovers" and was a Nominees.
Jeffrey Wright was nominated in Actor in a Leading Role for "American Fiction" and was a Nominees.


Robert Downey Jr. was nominated in Actor in a Supporting Role for "Oppenheimer" and was a Winner.
Sterling K. Brown was nominated in Actor in a Supporting Role for "American Fiction" and was a Nominees.
Robert De Niro was nominated in Actor in a Supporting Role for "Killers of the Flower Moon" and was a Nominees.
Ryan Gosling was nominated in Actor in a Supporting Role for "Barbie" and was a Nominees.
Mark Ruffalo was nominated in Actor in a Supporting Role for "Poor Things" and was a Nominees.


Emma Stone was nominated in Act

Rather than printing out the information, store it in `nominees_2018` so that we can turn it into a DataFrame.

In [123]:
nominees_l = []

# For each award group
for group in oscars_award_groups:

    # Get the name of the category
    category = group.find('div',{'class':'field--name-field-award-category-oscars'}).text.strip()

    # Get all the nominees in the group
    nominees = group.find_all('div',{'class':'paragraph--type--award-honoree'})
    
    # Loop through each nominee
    for nominee in nominees:
        
        # Get the honoree type
        nom_winner = nominee.find('div',{'class':'field--name-field-honoree-type'}).text.strip()
        
        # Get the name
        nom_name = nominee.find('div',{'class':'field--name-field-award-entities'}).text.strip()
        
        # Get the movie
        nom_movie = nominee.find('div',{'class':'field--name-field-award-film'}).text.strip()

        # Make a payload for each nominee
        nominee_payload = {
            'Category':category,
            'Name':nom_name,
            'Movie':nom_movie,
            'Year':2024, # We're only looking at 2018 right now
            'Winner':nom_winner
        }
        
        # Add the payload to the list of nominees at top
        nominees_l.append(nominee_payload)

Moment of truth!

In [124]:
nominees_df = pd.DataFrame(nominees_l)
nominees_df

,Category,Name,Movie,Year,Winner
0,Actor in a Leading Role,Cillian Murphy,Oppenheimer,2024,Winner
1,Actor in a Leading Role,Bradley Cooper,Maestro,2024,Nominees
2,Actor in a Leading Role,Colman Domingo,Rustin,2024,Nominees
3,Actor in a Leading Role,Paul Giamatti,The Holdovers,2024,Nominees
4,Actor in a Leading Role,Jeffrey Wright,American Fiction,2024,Nominees
...,...,...,...,...,...
115,Writing (Original Screenplay),Screenplay - Justine Triet and Arthur Harari,Anatomy of a Fall,2024,Winner
116,Writing (Original Screenplay),Written by David Hemingson,The Holdovers,2024,Nominees
117,Writing (Original Screenplay),Written by Bradley Cooper & Josh Singer,Maestro,2024,Nominees
118,Writing (Original Screenplay),Screenplay by Samy Burch; Story by Samy Burch ...,May December,2024,Nominees


In [126]:
nominees_df['Movie'].value_counts()

Movie
Oppenheimer                                          13
Poor Things                                          11
Killers of the Flower Moon                            9
Maestro                                               7
Barbie                                                6
Anatomy of a Fall                                     5
The Holdovers                                         5
American Fiction                                      5
The Zone of Interest                                  4
Napoleon                                              3
The Creator                                           2
Nyad                                                  2
Past Lives                                            2
Mission: Impossible - Dead Reckoning Part One         2
Godzilla Minus One                                    1
WAR IS OVER! Inspired by the Music of John & Yoko     1
Indiana Jones and the Dial of Destiny                 1
What Was I Made For?                      

Now let's turn this hulking beast of a parser into a function so we can apply it to other years' nominees in the next step.

In [174]:
def parse_nominees(oscars_award_groups,year):
    nominees_l = []

    # For each award group
    for group in oscars_award_groups:
    
        # Get the name of the category
        category = group.find('div',{'class':'field--name-field-award-category-oscars'}).text.strip()
    
        # Get all the nominees in the group
        nominees = group.find_all('div',{'class':'paragraph--type--award-honoree'})
        
        # Loop through each nominee
        for nominee in nominees:
            
            # Get the honoree type
            try:
                nom_winner = nominee.find('div',{'class':'field--name-field-honoree-type'}).text.strip()
            except:
                nom_winner = ''
            
            # Get the name
            try:
                nom_name = nominee.find('div',{'class':'field--name-field-award-entities'}).text.strip()
            except:
                nom_name = ''
            
            # Get the movie
            try:
                nom_movie = nominee.find('div',{'class':'field--name-field-award-film'}).text.strip()
            except:
                nom_movie = ''
    
            # Make a payload for each nominee
            nominee_payload = {
                'Category':category,
                'Name':nom_name,
                'Movie':nom_movie,
                'Year':year,
                'Winner':nom_winner
            }
            
            # Add the payload to the list of nominees at top
            nominees_l.append(nominee_payload)
    
    return nominees_l

In [130]:
parse_nominees(oscars_award_groups,2024)

[{'Category': 'Actor in a Leading Role',
  'Name': 'Cillian Murphy',
  'Movie': 'Oppenheimer',
  'Year': 2024,
  'Winner': 'Winner'},
 {'Category': 'Actor in a Leading Role',
  'Name': 'Bradley Cooper',
  'Movie': 'Maestro',
  'Year': 2024,
  'Winner': 'Nominees'},
 {'Category': 'Actor in a Leading Role',
  'Name': 'Colman Domingo',
  'Movie': 'Rustin',
  'Year': 2024,
  'Winner': 'Nominees'},
 {'Category': 'Actor in a Leading Role',
  'Name': 'Paul Giamatti',
  'Movie': 'The Holdovers',
  'Year': 2024,
  'Winner': 'Nominees'},
 {'Category': 'Actor in a Leading Role',
  'Name': 'Jeffrey Wright',
  'Movie': 'American Fiction',
  'Year': 2024,
  'Winner': 'Nominees'},
 {'Category': 'Actor in a Supporting Role',
  'Name': 'Robert Downey Jr.',
  'Movie': 'Oppenheimer',
  'Year': 2024,
  'Winner': 'Winner'},
 {'Category': 'Actor in a Supporting Role',
  'Name': 'Sterling K. Brown',
  'Movie': 'American Fiction',
  'Year': 2024,
  'Winner': 'Nominees'},
 {'Category': 'Actor in a Supporting R

## Iterating vs. parsing to retrieve data

Often the data you are interested in is spread across multiple web pages. In an ideal world, the naming conventions would let you retrieve the data from these pages systematically. In the case of the Oscars, the URLs appear to be consistently formatted: `https://www.oscars.org/oscars/ceremonies/2019` suggests that we could change the 2019 to any other date going back to the start of the Oscars and get that year as well: `https://www.oscars.org/oscars/ceremonies/2018` should get us the page for 2018, and so on. Let's demonstrate each of these strategies with the Oscars data: iterating from 2019 back to 1929 in the URL versus parsing the list of links from the header.

### Iterating strategies for retrieving data

The fundamental assumption with this strategy is that the data are stored at URLs in a consistent way that we can access sequentially. In the case of the Oscars, we *should* be able to simply pass each year to the URL in requests. Here we want to practice responsible data scraping by including a sleep between each request so that we do not overwhelm the Oscars server with requests. We can use the `sleep` function within `time`.

In [131]:
from time import sleep

The `sleep(3)` below prevents any more code from progressing for 3 seconds.

In [132]:
print("The start of something.")
sleep(5)
print("The end of something.")

The start of something.
The end of something.


The core part of the iterating strategy is simply using Python's [`range`](https://docs.python.org/3.7/library/functions.html#func-range) function to generate a sequence of values. Here, we can use `range` to print out a sequence of URLs that should correspond to awards pages from 2010 through 2019. We can also incorporate the `sleep` functionality and wait a second between each `print` statement—it should now take 10 seconds for this code to finish printing. This simulates how we can use `sleep` to slow down and spread out requests so that we do not overwhelm the servers whose data we are trying to scrape.

In [133]:
for year in range(2010,2025):
    sleep(1)
    print('https://www.oscars.org/oscars/ceremonies/{0}'.format(year))

https://www.oscars.org/oscars/ceremonies/2010
https://www.oscars.org/oscars/ceremonies/2011
https://www.oscars.org/oscars/ceremonies/2012
https://www.oscars.org/oscars/ceremonies/2013
https://www.oscars.org/oscars/ceremonies/2014
https://www.oscars.org/oscars/ceremonies/2015
https://www.oscars.org/oscars/ceremonies/2016
https://www.oscars.org/oscars/ceremonies/2017
https://www.oscars.org/oscars/ceremonies/2018
https://www.oscars.org/oscars/ceremonies/2019
https://www.oscars.org/oscars/ceremonies/2020
https://www.oscars.org/oscars/ceremonies/2021
https://www.oscars.org/oscars/ceremonies/2022
https://www.oscars.org/oscars/ceremonies/2023
https://www.oscars.org/oscars/ceremonies/2024


We defined a function `parse_nominees` above that takes the "true groups" of nominees. Let's try to tie these pieces together for all the nominees in the 2010s.

In [175]:
# Create an empty list to store the data we get
all_years_nominees = dict()

# For each year starting in 2010 until 2019
for year in range(1950,1960):
    
    # Pause for a second between each request
    sleep(1)
    
    # Get the raw HTML
    year_raw_html = requests.get('https://www.oscars.org/oscars/ceremonies/{0}'.format(year),headers=user_agent).text

    # Soup-ify
    oscars_soup = BeautifulSoup(year_raw_html)

    # Get the awards groups
    oscars_award_groups = oscars_soup.find_all('div',{'class':'paragraph--type--award-category'})
    
    # Use our parsing function, passing the year from above
    year_nominees = parse_nominees(oscars_award_groups,year)
    
    # Convert the year_nominees to a DataFrame and add them to all_years_nominees
    all_years_nominees[year] = pd.DataFrame(year_nominees)

Combine each of the DataFrames in `all_years_nominees` into a giant DataFrame of all the nominees from 2010-2019.

In [176]:
all_years_nominees_df = pd.concat(all_years_nominees)
all_years_nominees_df.reset_index(drop=True)

,Category,Name,Movie,Year,Winner
0,Actor,Broderick Crawford,All the King's Men,1950,Winner
1,Actor,Kirk Douglas,Champion,1950,Nominees
2,Actor,Gregory Peck,Twelve O'Clock High,1950,Nominees
3,Actor,Richard Todd,The Hasty Heart,1950,Nominees
4,Actor,John Wayne,Sands of Iwo Jima,1950,Nominees
...,...,...,...,...,...
1152,Writing (Story and Screenplay—written directly...,"Nedrick Young, Harold Jacob Smith",The Defiant Ones,1959,Winner
1153,Writing (Story and Screenplay—written directly...,Paddy Chayefsky,The Goddess,1959,Nominees
1154,Writing (Story and Screenplay—written directly...,"Melville Shavelson, Jack Rose",Houseboat,1959,Nominees
1155,Writing (Story and Screenplay—written directly...,Story by James Edward Grant; Screenplay by Wi...,The Sheepman,1959,Nominees


In [ ]:
all_years_nominees_df['Name'].value_counts()

### Parsing strategy for retrieving data

Frustratingly, this iterating strategy may not always hold: maybe some years are skipped or the naming convention changes at some point. We will cover some basics of [error-handling in Python](https://realpython.com/python-exceptions/) that could let us work around errors as they pop up, but this may result in an incomplete collection if the naming conventions are systematic. What we would want to do is to identify all the links ahead of time by parsing them from list and then work through that list to get the complete data collection.

What this means in the context of our Oscars example is assuming that we cannot trust that the sequential numbering of the years is a realiable guide to get all the data. Instead, we should get a list of the URLs for each of the awards pages from the "ceremonies-decade-scroller" (from Inspect) at the top. This scroller *should* be consistent across all the pages, but start with the nominees for 2019 just to be safe:

In [ ]:
oscars2019_raw = requests.get('https://www.oscars.org/oscars/ceremonies/2019',headers=user_agent).text
oscars2019_soup = BeautifulSoup(oscars2019_raw)

Using the Inspect tool, there is a `<div class="years">` that contains the links to each of the years. Run a `.find_all` to get all these href locations.

In [ ]:
# Get the <div class="years"> as a parent tag first, just in case there are <a class="years"> elsewhere
oscars2019_years_div = oscars2019_soup.find_all('div',{'class':'years'})[0]

# Now get the <a class="years"> underneath only the oscars2019_years_div
oscars2019_years_a = oscars2019_years_div.find_all('a',{'class':'year'})

# Inspect the first 10
oscars2019_years_a[:10]

Each of these `<a>` tags contains an "href", or the URL element where the page lives, and a text element for what's displayed.

In [ ]:
oscars2019_years_a[0]['href']

In [ ]:
oscars2019_years_a[0].text

Now we can write a loop to print out the URL locations for all the other award years based on the "official" links in the "ceremonies-decade-scroller" navigation rather than assuming the years are sequential—I promise this will pay dividends in the future when inconsistent design wreaks havoc on your sequential data strategies!

In [ ]:
for a in oscars2019_years_a[-10:]:
    href = a['href']
    print('https://www.oscars.org' + href)

We can now use the `parse_nominees` function for these pages as well.

In [ ]:
# Create an empty list to store the data we get
all_years_nominees = dict()

# For the 10 most recent years
for a in oscars2019_years_a[-10:]:
    
    # Pause for a second between each request
    sleep(1)
    
    # Get the href
    href = a['href']
    
    # Get the year
    year = a.text
    
    # Get the raw HTML
    url = 'https://www.oscars.org' + href
    year_raw_html = requests.get(url,headers=user_agent).text

    # Soup-ify
    year_souped_html = BeautifulSoup(year_raw_html)
    
    # Get the parent group
    year_parent_group = year_souped_html.find_all('div',{'id':'quicktabs-tabpage-honorees-0'})
    
    # Get the true groups under the parent group
    year_true_groups = year_parent_group[0].find_all('div',{'class':'view-grouping'})
    
    # Use our parsing function, passing the year from above
    year_nominees = parse_nominees(year_true_groups,year)
    
    # Convert the year_nominees to a DataFrame and add them to all_years_nominees
    all_years_nominees[year] = pd.DataFrame(year_nominees)

Combine each of the DataFrames in `all_years_nominees` into a giant DataFrame of all the nominees from 2010-2019.

In [ ]:
all_years_nominees_df = pd.concat(all_years_nominees)
all_years_nominees_df.reset_index(drop=True).head(10)

In [ ]:
all_years_nominees_df['Winner'].value_counts()

## Exercises

Let's take a look at websites you're interested in scraping and see what kinds of challenges and opportunities exist for scraping their data.

### Exercise 01: Retrieve a table from a Wikipedia article

Retrieve the "Tallest buildings in the world" table from the "[List of tallest buildings](https://en.wikipedia.org/wiki/List_of_tallest_buildings)" Wikipedia article.

### Exercise 02: Retrieve the top-grossing movies of 2022
Retrieve the "[Top Grossing Movies of 2022](https://www.the-numbers.com/market/2022/summary)" table from The Numbers and parse into a DataFrame.

### Exercise 03: Retrieve corresponding Wikipedia pageviews
For each of the top 10 grossing movies of 2022, retrieve the corresponding Wikipedia article pageview counts ([docs](https://wikimedia.org/api/rest_v1/)) for January 1, 2022 through December 31, 2022.

In [ ]:
from urllib.parse import quote 

def get_pageviews(article_name,start='20220101',end='20221231'):
    article_title = quote(article_name)

    wiki_pvs_url = 'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia.org/all-access/user/{0}/daily/{1}/{2}'

    custom_headers = {'User-Agent':'cu-info-5871/0.0 (brian.keegan@colorado.edu)'}

    req = requests.get(
        url = wiki_pvs_url.format(article_title,start,end),
        headers = custom_headers
    )

    # TODO: Don't just return the JSON, you have to navigate into the JSON
    return req.json()

### Exercise 04: Compare box office performance with Wikipedia activity
Aggregate the Wikipedia pageview data into a total count of pageviews for the year. Combine the total pageview data with the total movie revenue data. Make a visualization to compare these two types of data.

### Exercise 05: Retrieve tweet data (hard!)
**This exercise may involve breaking X/Twitter's Terms of Service. Retrieving data about official accounts of public officials *should be* a fair use exception to this rule.**

Inspect the source code for the user timeline for the [@POTUS](https://twitter.com/potus) or [@WhiteHouse](https://twitter.com/WhiteHouse) X/Twitter accounts. Try to retrieve the content of the tweets on the account's timeline.

In [143]:
raw_html = requests.get('https://leg.colorado.gov/bills/hb24-1147',headers=user_agent).text

raw_html

'<!DOCTYPE html>\n<!--[if IEMobile 7]><html class="iem7"  lang="en" dir="ltr"><![endif]-->\n<!--[if lte IE 6]><html class="lt-ie9 lt-ie8 lt-ie7 no-js"  lang="en" dir="ltr"><![endif]-->\n<!--[if (IE 7)&(!IEMobile)]><html class="lt-ie9 lt-ie8 no-js"  lang="en" dir="ltr"><![endif]-->\n<!--[if IE 8]><html class="lt-ie9 no-js"  lang="en" dir="ltr"><![endif]-->\n<!--[if IE 9]><html class="ie9 lt-ie10 no-js"  lang="en" dir="ltr"><![endif]-->\n<!--[if (gt IE 9)|(gt IEMobile 7)]><!--><html class="no-js"  lang="en" dir="ltr" prefix="content: http://purl.org/rss/1.0/modules/content/ dc: http://purl.org/dc/terms/ foaf: http://xmlns.com/foaf/0.1/ og: http://ogp.me/ns# rdfs: http://www.w3.org/2000/01/rdf-schema# sioc: http://rdfs.org/sioc/ns# sioct: http://rdfs.org/sioc/types# skos: http://www.w3.org/2004/02/skos/core# xsd: http://www.w3.org/2001/XMLSchema#"><!--<![endif]-->\n\n<head>\n  <meta charset="utf-8" /><script type="text/javascript">(window.NREUM||(NREUM={})).init={ajax:{deny_list:["bam.nr-

In [144]:
leg_soup = BeautifulSoup(raw_html)

In [149]:
leg_soup.find('div',{'id':"block-cga-bills-bill-sponsors"}).find_all('div',{'class':'member'})

[<div class="member">
 <div class="member-image">
 <a href="/legislators/junie-joseph"><img alt="Representative" src="https://leg.colorado.gov/sites/default/files/styles/profile_picture_teaser/public/2024b_joseph%2Cjunie.jpg?itok=cLoUsKep"/></a> </div>
 <div class="member-details">
 <span class="member-title">Representative</span>
 <h4><a href="/legislators/junie-joseph">Junie Joseph</a></h4>
 </div>
 </div>,
 <div class="member">
 <div class="member-image">
 <a href="/legislators/brianna-titone"><img alt="Representative" src="https://leg.colorado.gov/sites/default/files/styles/profile_picture_teaser/public/2024b_titone%2Cbrianna.jpg?itok=YzCwDK_Z"/></a> </div>
 <div class="member-details">
 <span class="member-title">Representative</span>
 <h4><a href="/legislators/brianna-titone">Brianna Titone</a></h4>
 </div>
 </div>,
 <div class="member">
 <div class="member-image">
 <a href="/legislators/chris-hansen"><img alt="Senator" src="https://leg.colorado.gov/sites/default/files/styles/pro

In [150]:
leg_soup.find('div',{'id':"block-cga-bills-bill-committees"})

<div class="block block-cga-bills last odd" id="block-cga-bills-bill-committees">
<div class="block-container">
<div class="block-title-container">
<h2 class="block__title block-title">Committees</h2>
</div>
<div class="block-content">
<div class="committee-item">
<div class="small-label">
      House    </div>
<h4><a href="/committees/state-civic-military-veterans-affairs/2024-regular-session">State, Civic, Military, &amp; Veterans Affairs</a></h4>
<div class="small-label">
      Senate    </div>
<h4><a href="/committees/state-veterans-military-affairs/2024-regular-session">State, Veterans, &amp; Military Affairs</a></h4>
</div> </div>
</div>
</div>

## Final project down payment

1. Find a legislator from the list of current Colorado legislators: https://leg.colorado.gov/legislators/
2. `requests.get` their biograpaphy page
3. Parse out some of their basic demographic information: party, occupation, district, committees
4. Store this information in a dictionary

In [162]:
# Get the page
joseph_raw = requests.get('https://leg.colorado.gov/legislators/brianna-titone').text

# Turn into soup
joseph_soup = BeautifulSoup(joseph_raw)

# Find party affiliation
joseph_soup.find('div',{'class':'field-name-field-political-affiliation'}).find('div',{'class':'even'}).text

'Democrat'

In [164]:
joseph_soup.find('div',{'class':'field-name-field-leadership-position'}).find('div',{'class':'even'}).text

'Majority Caucus Co-Chair'

In [166]:
joseph_soup.find('h1').text

'Brianna Titone'

In [170]:
joseph_soup.find_all('div',{'class':'committee-assignment'})[0].h4.text

'Joint Technology Committee'

In [171]:
joseph_soup.find_all('div',{'class':'committee-assignment'})[1].h4.text

'Agriculture, Water & Natural Resources'

In [172]:
joseph_soup.find_all('div',{'class':'committee-assignment'})[2].h4.text

'Energy & Environment'

In [173]:
joseph_soup.find_all('div',{'class':'committee-assignment'})[3].h4.text

IndexError: list index out of range